# Figure 2: Diagnose-step graph (full version + paper inset)

One layout, two outputs:
* `fig_graph_full.pdf`: every stable link that touches the label or the declared labeling-process variable (GitHub supplement).
* `fig_graph_inset.pdf`: a crop of the same drawing (paper). Same positions, fonts and styles, so the supplement looks familiar.

**Input:** the stable-edge list below is transcribed from the current Figure 2 (NHANES training split, edges in at least 80% of 50 bootstrap resamples). Before submission, replace it with the edge output of the Diagnose notebook so the figure is generated from the run itself.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

plt.rcParams.update({"font.family": "sans-serif", "font.size": 7, "pdf.fonttype": 42})

## 1. Input: stable links that touch the label or the declared variable
Three kinds of attribute, as in Section 6.1:
* linked only to the cholesterol check (the O/E check is valid),
* linked to both (filtered by the label-link filter),
* linked only to the label.

In [ ]:
CHK, LAB = "chk", "lab"

check_only = ["usual place for care", "has insurance", "smoker", "income"]
both       = ["doctor visit (past yr)", "education", "Hispanic",
              "age", "BMI", "high cholesterol", "high blood pressure"]
label_only = ["general health (1 = excellent)", "stroke", "heart disease / attack", "Black", "male"]

# variables whose groups were flagged (bold text, as in the original figure)
flagged = {"usual place for care", "smoker", CHK}

edges = ([(v, CHK) for v in check_only]
         + [(v, CHK) for v in both] + [(v, LAB) for v in both]
         + [(v, LAB) for v in label_only]
         + [(CHK, LAB)])
print(len(check_only), len(both), len(label_only), "attributes;", len(edges), "edges")

## 2. Layout
Left to right: check-only attributes | declared variable | attributes linked to both (above and below the main link) | label | label-only attributes.
The inset is the crop `x < INSET_XMAX`, which drops the label-only column.

In [ ]:
pos = {CHK: (2.6, 0.0), LAB: (9.4, 0.0)}
for v, y in zip(check_only, [1.5, 0.5, -0.5, -1.5]):
    pos[v] = (0.0, y)
for v, x in zip(both[:3], [4.6, 6.2, 7.8]):
    pos[v] = (x, 1.5)
for v, x in zip(both[3:], [4.2, 5.3, 6.5, 8.0]):
    pos[v] = (x, -1.5)
for v, y in zip(label_only, [1.6, 0.8, 0.0, -0.8, -1.6]):
    pos[v] = (12.4, y)

names = {CHK: "cholesterol check (5 yr)\n(labeling-process variable)", LAB: "told: diabetes\n(label)",
         "doctor visit (past yr)": "doctor visit\n(past yr)", "high cholesterol": "high\ncholesterol",
         "high blood pressure": "high blood\npressure", "usual place for care": "usual place\nfor care",
         "general health (1 = excellent)": "general health\n(1 = excellent)",
         "heart disease / attack": "heart disease\n/ attack"}
INSET_XMAX = 10.4

## 3. Drawing function (shared by both outputs)

In [ ]:
def draw(ax, show_label_only=True):
    keep = lambda v: show_label_only or v not in label_only
    # shaded bands: where the O/E check is applied vs filtered
    ax.add_patch(FancyBboxPatch((-1.0, -2.05), 2.0, 4.1, boxstyle="round,pad=0.02",
                                fc="#e8f1e8", ec="none", zorder=0))
    ax.add_patch(FancyBboxPatch((3.6, -2.05), 5.2, 4.1, boxstyle="round,pad=0.02",
                                fc="#eeeeee", ec="none", zorder=0))
    ax.text(0.0, 2.25, "linked only to the declared variable:\nO/E check applied",
            ha="center", va="bottom", fontsize=6.5, style="italic")
    ax.text(6.25, 2.25, "also linked to the label:\nremoved by the label-link filter",
            ha="center", va="bottom", fontsize=6.5, style="italic")
    if show_label_only:
        ax.text(12.4, 2.25, "linked only to the label", ha="center", va="bottom",
                fontsize=6.5, style="italic")
    # edges
    for a, b in edges:
        if not (keep(a) and keep(b)):
            continue
        (x1, y1), (x2, y2) = pos[a], pos[b]
        main = {a, b} == {CHK, LAB}
        ax.plot([x1, x2], [y1, y2], color="black" if main else "0.6",
                lw=1.6 if main else 0.6, zorder=1)
    # nodes
    for v, (x, y) in pos.items():
        if not keep(v):
            continue
        core = v in (CHK, LAB)
        ax.text(x, y, names.get(v, v), ha="center", va="center", zorder=3,
                fontweight="bold" if v in flagged else "normal",
                bbox=dict(boxstyle="round,pad=0.3", fc="white",
                          ec="black", lw=1.2 if core else 0.5))
    ax.set_aspect("equal")
    ax.axis("off")

## 4. Output: full figure (supplement)

In [ ]:
fig, ax = plt.subplots(figsize=(9.0, 2.9))
draw(ax, show_label_only=True)
ax.set_xlim(-1.3, 13.6); ax.set_ylim(-2.2, 3.0)
fig.savefig("fig_graph_full.pdf", bbox_inches="tight")
fig.savefig("fig_graph_full.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Output: paper inset (same drawing, cropped)

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 2.6))
draw(ax, show_label_only=False)
ax.set_xlim(-1.3, INSET_XMAX); ax.set_ylim(-2.2, 3.0)
fig.savefig("fig_graph_inset.pdf", bbox_inches="tight")
fig.savefig("fig_graph_inset.png", dpi=200, bbox_inches="tight")
plt.show()